In [12]:

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("proyecto-agua-calidad-datos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

In [14]:
ORIGEN_DATOS = "/opt/s03-procesamiento-calidad-datos/data"
ARTIFACTS = "/opt/s03-procesamiento-calidad-datos/artifacts"

In [15]:
!find /opt -name "mediciones_calidad_agua.parquet" 2>/dev/null


/opt/s03 - Procesamiento y Calidad de Datos/data/mediciones_calidad_agua.parquet


In [16]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

# Ruta correcta dentro del entorno donde se ejecuta Spark
ORIGEN_DATOS = "/opt/s03 - Procesamiento y Calidad de Datos/data"

schema_agua = StructType([
    StructField("id_lectura", IntegerType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("ubicacion", StringType(), True),
    StructField("fecha_hora", TimestampType(), True),
    StructField("canal_transmision_id", IntegerType(), True),
    StructField("ph", DoubleType(), True),
    StructField("turbidez_ntu", DoubleType(), True),
    StructField("temperatura_c", DoubleType(), True),
    StructField("conductividad_us_cm", DoubleType(), True),
    StructField("solidos_disueltos_totales_mg_l", DoubleType(), True),
    StructField("oxigeno_disuelto_mg_l", DoubleType(), True),
    StructField("plomo_mg_l", DoubleType(), True),
    StructField("arsenico_mg_l", DoubleType(), True),
    StructField("mercurio_mg_l", DoubleType(), True),
    StructField("cadmio_mg_l", DoubleType(), True),
    StructField("coliformes_fecales_nmp_100ml", DoubleType(), True),
    StructField("escherichia_coli_nmp_100ml", DoubleType(), True),
    StructField("presencia_parasitos", IntegerType(), True),
    StructField("radiactividad_bq_l", DoubleType(), True),
    StructField("caudal_l_s", DoubleType(), True),
    StructField("indice_riesgo_normalizado", DoubleType(), True),
])

df_agua = spark.read.parquet(
    f"{ORIGEN_DATOS}/mediciones_calidad_agua.parquet"
)

df_agua.printSchema()

root
 |-- medicion_id: long (nullable = true)
 |-- estacion_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- hora: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- ph: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- oxigeno_disuelto_mgl: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_ppm: double (nullable = true)
 |-- nitratos_mgl: double (nullable = true)
 |-- fosfatos_mgl: double (nullable = true)
 |-- coliformes_fecales_cfu: double (nullable = true)
 |-- observaciones: string (nullable = true)



In [18]:
columnas_requeridas = {
    "id_lectura", "sensor_id", "ubicacion", "fecha_hora",
    "ph", "plomo_mg_l", "arsenico_mg_l", "mercurio_mg_l",
    "presencia_parasitos", "radiactividad_bq_l",
}

faltantes = columnas_requeridas - set(df_agua.columns)

if faltantes:
    print(f"Faltan columnas obligatorias: {sorted(faltantes)}")
else:
    print("Esquema validado: las columnas requeridas están presentes.")

print("\nColumnas disponibles:")
print(df_agua.columns)

print("\nCantidad de registros:")
print(df_agua.count())


Faltan columnas obligatorias: ['arsenico_mg_l', 'fecha_hora', 'id_lectura', 'mercurio_mg_l', 'plomo_mg_l', 'presencia_parasitos', 'radiactividad_bq_l', 'sensor_id', 'ubicacion']

Columnas disponibles:
['medicion_id', 'estacion_id', 'fecha', 'hora', 'canal', 'ph', 'temperatura_c', 'turbidez_ntu', 'oxigeno_disuelto_mgl', 'conductividad_us_cm', 'solidos_disueltos_ppm', 'nitratos_mgl', 'fosfatos_mgl', 'coliformes_fecales_cfu', 'observaciones']

Cantidad de registros:
1250000


In [19]:
from pyspark.sql.functions import col, count, when

total_filas = df_agua.count()

nulos = df_agua.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_agua.columns
]).collect()[0].asDict()

for columna, cantidad in nulos.items():
    porcentaje = cantidad / total_filas * 100
    print(f"{columna}: {cantidad} nulos ({porcentaje:.1f}%)")

medicion_id: 0 nulos (0.0%)
estacion_id: 0 nulos (0.0%)
fecha: 0 nulos (0.0%)
hora: 0 nulos (0.0%)
canal: 0 nulos (0.0%)
ph: 0 nulos (0.0%)
temperatura_c: 0 nulos (0.0%)
turbidez_ntu: 0 nulos (0.0%)
oxigeno_disuelto_mgl: 0 nulos (0.0%)
conductividad_us_cm: 0 nulos (0.0%)
solidos_disueltos_ppm: 0 nulos (0.0%)
nitratos_mgl: 1099848 nulos (88.0%)
fosfatos_mgl: 1099848 nulos (88.0%)
coliformes_fecales_cfu: 1099848 nulos (88.0%)
observaciones: 0 nulos (0.0%)


In [21]:
from pyspark.sql.functions import col, trim

df_agua.filter(
    col("estacion_id").isNull() | (trim(col("estacion_id")) == "")
).count()


0

In [23]:
from pyspark.sql.functions import col

df_agua.filter("estacion_id IS NOT NULL").show(5, truncate=False)

df_agua.filter("ph BETWEEN 6.5 AND 8.5").count()

df_agua.filter(
    col("estacion_id").isNotNull()
).show(5, truncate=False)

df_agua.filter(
    col("ph").between(6.5, 8.5)
).count()


+-----------+-----------+----------+-----+----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+-------------+
|medicion_id|estacion_id|fecha     |hora |canal     |ph  |temperatura_c|turbidez_ntu|oxigeno_disuelto_mgl|conductividad_us_cm|solidos_disueltos_ppm|nitratos_mgl|fosfatos_mgl|coliformes_fecales_cfu|observaciones|
+-----------+-----------+----------+-----+----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+-------------+
|1          |EST-0133   |2025-08-22|04:30|sensor_iot|7.8 |18.8         |3.93        |6.76                |557.1              |341.0                |NULL        |NULL        |NULL                  |             |
|2          |EST-0063   |2024-11-12|10:35|sensor_iot|6.29|8.2          |10.34       |7.0                 |431.8              |243.2                |NULL

1081689

In [25]:
print("between (incluye 6.5 y 8.5):", df_agua.filter(col("ph").between(6.5, 8.5)).count())
print("estricto (excluye 6.5 y 8.5):", df_agua.filter((col("ph") > 6.5) & (col("ph") < 8.5)).count())

between (incluye 6.5 y 8.5): 1081689
estricto (excluye 6.5 y 8.5): 1076605


In [27]:
# Comprobar valores NULL en la columna ph
df_agua.filter(
    col("ph").eqNullSafe(None)
).count()

0

In [28]:
# Cantidad de valores NULL por columna

print("Valores NULL en ph:")
print(df_agua.filter(col("ph").eqNullSafe(None)).count())

print("Valores NULL en temperatura_c:")
print(df_agua.filter(col("temperatura_c").eqNullSafe(None)).count())

print("Valores NULL en turbidez_ntu:")
print(df_agua.filter(col("turbidez_ntu").eqNullSafe(None)).count())

print("Valores NULL en oxigeno_disuelto_mgl:")
print(df_agua.filter(col("oxigeno_disuelto_mgl").eqNullSafe(None)).count())

print("Valores NULL en observaciones:")
print(df_agua.filter(col("observaciones").eqNullSafe(None)).count())


Valores NULL en ph:
0
Valores NULL en temperatura_c:
0
Valores NULL en turbidez_ntu:
0
Valores NULL en oxigeno_disuelto_mgl:
0
Valores NULL en observaciones:
0


In [29]:
df_agua.filter(col("ph").eqNullSafe(None)).count()


0

In [30]:
df_agua.select("ph").show(10)


+----+
|  ph|
+----+
| 7.8|
|6.29|
|6.44|
|6.98|
|7.51|
|7.55|
|6.67|
|7.96|
|5.96|
|7.33|
+----+
only showing top 10 rows


In [32]:
from pyspark.sql.functions import col

df_agua.orderBy(
    col("turbidez_ntu").desc()
).show(5, truncate=False)

df_agua.orderBy(
    col("estacion_id").asc(),
    col("ph").desc()
).show(5, truncate=False)


df_agua.sort(
    col("ph").asc_nulls_last()
).show(5, truncate=False)


+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+-----------------------------------+
|medicion_id|estacion_id|fecha     |hora |canal      |ph  |temperatura_c|turbidez_ntu|oxigeno_disuelto_mgl|conductividad_us_cm|solidos_disueltos_ppm|nitratos_mgl|fosfatos_mgl|coliformes_fecales_cfu|observaciones                      |
+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+-----------------------------------+
|201308     |EST-0172   |2025-05-20|11:43|sensor_iot |8.43|21.3         |94.97       |5.6                 |277.7              |193.6                |NULL        |NULL        |NULL                  |                                   |
|585901     |EST-0058   |2024-03-02|01:28|laboratorio|8.22|1

In [33]:
from pyspark.sql.functions import count as count_

df_agua.groupBy("medicion_id") \
    .count() \
    .filter("count > 1") \
    .show()


df_agua.groupBy(
    "estacion_id",
    "fecha",
    "hora"
).count() \
 .filter("count > 1") \
 .show()

+-----------+-----+
|medicion_id|count|
+-----------+-----+
+-----------+-----+

+-----------+----------+-----+-----+
|estacion_id|     fecha| hora|count|
+-----------+----------+-----+-----+
|   EST-0130|2025-11-14|01:45|    2|
|   EST-0080|2024-10-30|23:56|    2|
|   EST-0076|2024-04-24|03:43|    2|
|   EST-0023|2025-11-04|15:56|    2|
|   EST-0002|2024-04-30|15:50|    2|
|   EST-0017|2024-10-09|17:39|    2|
|   EST-0031|2024-08-28|09:43|    2|
|   EST-0154|2025-06-26|01:49|    2|
|   EST-0135|2025-08-04|08:44|    2|
|   EST-0180|2024-07-30|02:03|    2|
|   EST-0055|2024-07-15|17:19|    2|
|   EST-0043|2025-11-23|15:27|    2|
|   EST-0052|2024-07-28|17:48|    2|
|   EST-0135|2025-12-14|08:52|    2|
|   EST-0046|2025-06-28|13:14|    2|
|   EST-0105|2025-10-20|10:34|    2|
|   EST-0115|2024-09-05|07:33|    2|
|   EST-0106|2025-05-24|12:38|    2|
|   EST-0132|2025-05-05|08:54|    2|
|   EST-0150|2025-05-21|05:39|    2|
+-----------+----------+-----+-----+
only showing top 20 rows


In [34]:
total = df_agua.count()

# Elimina duplicados considerando toda la fila
sin_dup_fila_completa = df_agua.distinct().count()

# Elimina duplicados según el identificador de medición
sin_dup_por_medicion = df_agua.dropDuplicates(
    ["medicion_id"]
).count()

# Elimina duplicados según estación + fecha + hora
sin_dup_estacion_fecha_hora = df_agua.dropDuplicates(
    ["estacion_id", "fecha", "hora"]
).count()


print(f"Total: {total}")
print(f"Sin duplicar (fila completa): {sin_dup_fila_completa}")
print(f"Sin duplicar (por medicion_id): {sin_dup_por_medicion}")
print(f"Sin duplicar (por estacion_id+fecha+hora): {sin_dup_estacion_fecha_hora}")


Total: 1250000
Sin duplicar (fila completa): 1250000
Sin duplicar (por medicion_id): 1250000
Sin duplicar (por estacion_id+fecha+hora): 1245817


In [35]:
# ============================================================
# TÉCNICA 2 — Window + row_number()
# ============================================================

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


# Creamos una ventana por estación + fecha + hora.
# Conservamos la medición con mayor medicion_id.

window_spec = Window \
    .partitionBy(
        "estacion_id",
        "fecha",
        "hora"
    ) \
    .orderBy(
        col("medicion_id").desc()
    )


# Numeramos las filas dentro de cada grupo
df_ranked = df_agua.withColumn(
    "row_num",
    row_number().over(window_spec)
)


# Conservamos solamente la primera fila de cada grupo
df_sin_duplicados = df_ranked \
    .filter(col("row_num") == 1) \
    .drop("row_num")


print(
    f"Filas originales: {df_agua.count()}, "
    f"tras Window+row_number(): {df_sin_duplicados.count()}"
)

[Stage 87:================================================>         (5 + 1) / 6]

Filas originales: 1250000, tras Window+row_number(): 1245817


In [36]:
from pyspark.sql.functions import count as count_

df_agua.groupBy("medicion_id") \
    .count() \
    .filter("count > 1") \
    .show()

df_agua.groupBy(
    "estacion_id",
    "fecha",
    "hora"
).count() \
 .filter("count > 1") \
 .show()

+-----------+-----+
|medicion_id|count|
+-----------+-----+
+-----------+-----+



[Stage 98:================================================>         (5 + 1) / 6]

+-----------+----------+-----+-----+
|estacion_id|     fecha| hora|count|
+-----------+----------+-----+-----+
|   EST-0130|2025-11-14|01:45|    2|
|   EST-0080|2024-10-30|23:56|    2|
|   EST-0076|2024-04-24|03:43|    2|
|   EST-0023|2025-11-04|15:56|    2|
|   EST-0002|2024-04-30|15:50|    2|
|   EST-0017|2024-10-09|17:39|    2|
|   EST-0031|2024-08-28|09:43|    2|
|   EST-0154|2025-06-26|01:49|    2|
|   EST-0135|2025-08-04|08:44|    2|
|   EST-0180|2024-07-30|02:03|    2|
|   EST-0055|2024-07-15|17:19|    2|
|   EST-0043|2025-11-23|15:27|    2|
|   EST-0052|2024-07-28|17:48|    2|
|   EST-0135|2025-12-14|08:52|    2|
|   EST-0046|2025-06-28|13:14|    2|
|   EST-0105|2025-10-20|10:34|    2|
|   EST-0115|2024-09-05|07:33|    2|
|   EST-0106|2025-05-24|12:38|    2|
|   EST-0132|2025-05-05|08:54|    2|
|   EST-0150|2025-05-21|05:39|    2|
+-----------+----------+-----+-----+
only showing top 20 rows


In [37]:
total = df_agua.count()

sin_dup_fila_completa = df_agua.distinct().count()

sin_dup_por_medicion = df_agua.dropDuplicates(
    ["medicion_id"]
).count()

sin_dup_estacion_fecha_hora = df_agua.dropDuplicates(
    ["estacion_id", "fecha", "hora"]
).count()

print(f"Total: {total}")
print(f"Sin duplicar (fila completa): {sin_dup_fila_completa}")
print(f"Sin duplicar (por medicion_id): {sin_dup_por_medicion}")
print(f"Sin duplicar (por estacion_id+fecha+hora): {sin_dup_estacion_fecha_hora}")


Total: 1250000
Sin duplicar (fila completa): 1250000
Sin duplicar (por medicion_id): 1250000
Sin duplicar (por estacion_id+fecha+hora): 1245817


In [38]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window \
    .partitionBy(
        "estacion_id",
        "fecha",
        "hora"
    ) \
    .orderBy(
        col("medicion_id").desc()
    )

df_ranked = df_agua.withColumn(
    "row_num",
    row_number().over(window_spec)
)

df_sin_duplicados = df_ranked \
    .filter(col("row_num") == 1) \
    .drop("row_num")

print(
    f"Filas originales: {df_agua.count()}, "
    f"tras Window+row_number(): {df_sin_duplicados.count()}"
)


[Stage 125:===============================================>         (5 + 1) / 6]

Filas originales: 1250000, tras Window+row_number(): 1245817


In [39]:
df_agua.printSchema()


root
 |-- medicion_id: long (nullable = true)
 |-- estacion_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- hora: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- ph: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- oxigeno_disuelto_mgl: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_ppm: double (nullable = true)
 |-- nitratos_mgl: double (nullable = true)
 |-- fosfatos_mgl: double (nullable = true)
 |-- coliformes_fecales_cfu: double (nullable = true)
 |-- observaciones: string (nullable = true)



In [40]:
df_agua_limpio = df_agua.na.fill({
    "canal": -1
})


In [41]:
df_agua_limpio = df_agua.na.fill({
    "canal": "DESCONOCIDO"
})


In [42]:
df_agua_limpio = df_agua


In [43]:
print(
    "Filas si se usa na.drop() sin argumentos:",
    df_agua.na.drop().count()
)


Filas si se usa na.drop() sin argumentos: 150152


In [44]:
df_agua_valido = df_agua_limpio.na.drop(
    subset=["estacion_id"]
)

print(
    f"Filas antes: {df_agua.count()}, "
    f"después de na.drop(subset=['estacion_id']): "
    f"{df_agua_valido.count()}"
)


Filas antes: 1250000, después de na.drop(subset=['estacion_id']): 1250000


In [45]:
assert df_agua_valido.filter(
    col("estacion_id").isNull()
).count() == 0

print("Validación correcta: no existen registros con estacion_id NULL.")


Validación correcta: no existen registros con estacion_id NULL.


In [46]:
df_agua_valido = df_agua_valido.cache()

print("DataFrame Silver cacheado correctamente.")


DataFrame Silver cacheado correctamente.


In [47]:
df_agua_valido.count()


1250000

In [48]:
df_agua_valido.show(10, truncate=False)


+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+--------------------------------------------------------+
|medicion_id|estacion_id|fecha     |hora |canal      |ph  |temperatura_c|turbidez_ntu|oxigeno_disuelto_mgl|conductividad_us_cm|solidos_disueltos_ppm|nitratos_mgl|fosfatos_mgl|coliformes_fecales_cfu|observaciones                                           |
+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+--------------------------------------------------------+
|1          |EST-0133   |2025-08-22|04:30|sensor_iot |7.8 |18.8         |3.93        |6.76                |557.1              |341.0                |NULL        |NULL        |NULL                  |                                  

In [49]:
# ============================================================
# 10. ESCRITURA EN MÚLTIPLES FORMATOS (MUESTRA)
# ============================================================

# Tomar una muestra de 100 registros
muestra = df_agua_valido.limit(100)

print(f"Cantidad de registros de la muestra: {muestra.count()}")

# Mostrar algunos registros
muestra.show(5, truncate=False)

# ------------------------------------------------------------
# Guardar en CSV
# ------------------------------------------------------------
muestra.write \
    .format("csv") \
    .option("header", True) \
    .mode("overwrite") \
    .save(f"{ARTIFACTS}/muestra_csv")

# ------------------------------------------------------------
# Guardar en JSON
# ------------------------------------------------------------
muestra.write \
    .format("json") \
    .mode("overwrite") \
    .save(f"{ARTIFACTS}/muestra_json")

# ------------------------------------------------------------
# Guardar en Parquet
# ------------------------------------------------------------
muestra.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(f"{ARTIFACTS}/muestra_parquet")

print("============================================")
print("Muestras guardadas correctamente:")
print(f"CSV:     {ARTIFACTS}/muestra_csv")
print(f"JSON:    {ARTIFACTS}/muestra_json")
print(f"Parquet: {ARTIFACTS}/muestra_parquet")
print("============================================")


Cantidad de registros de la muestra: 100
+-----------+-----------+----------+-----+----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+-------------+
|medicion_id|estacion_id|fecha     |hora |canal     |ph  |temperatura_c|turbidez_ntu|oxigeno_disuelto_mgl|conductividad_us_cm|solidos_disueltos_ppm|nitratos_mgl|fosfatos_mgl|coliformes_fecales_cfu|observaciones|
+-----------+-----------+----------+-----+----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+-------------+
|1          |EST-0133   |2025-08-22|04:30|sensor_iot|7.8 |18.8         |3.93        |6.76                |557.1              |341.0                |NULL        |NULL        |NULL                  |             |
|2          |EST-0063   |2024-11-12|10:35|sensor_iot|6.29|8.2          |10.34       |7.0                 |431.8

In [51]:
df_agua_valido.select("estacion_id").distinct().show()


+-----------+
|estacion_id|
+-----------+
|   EST-0128|
|   EST-0043|
|   EST-0022|
|   EST-0167|
|   EST-0003|
|   EST-0130|
|   EST-0108|
|   EST-0110|
|   EST-0095|
|   EST-0074|
|   EST-0069|
|   EST-0005|
|   EST-0077|
|   EST-0147|
|   EST-0073|
|   EST-0154|
|   EST-0067|
|   EST-0053|
|   EST-0157|
|   EST-0063|
+-----------+
only showing top 20 rows


In [52]:
# ============================================================
# VERIFICAR COLUMNAS DE estaciones_agua.csv
# ============================================================

ruta_estaciones = f"{ORIGEN_DATOS}/estaciones_agua.csv"

df_estaciones = spark.read.csv(
    ruta_estaciones,
    header=True,
    inferSchema=True
)

print("Columnas de estaciones_agua.csv:")
print(df_estaciones.columns)

print("\nEsquema:")
df_estaciones.printSchema()

print("\nPrimeros registros:")
df_estaciones.show(10, truncate=False)


Columnas de estaciones_agua.csv:
['estacion_id', 'nombre_estacion', 'region', 'cuenca', 'tipo_fuente', 'latitud', 'longitud', 'fecha_instalacion']

Esquema:
root
 |-- estacion_id: string (nullable = true)
 |-- nombre_estacion: string (nullable = true)
 |-- region: string (nullable = true)
 |-- cuenca: string (nullable = true)
 |-- tipo_fuente: string (nullable = true)
 |-- latitud: double (nullable = true)
 |-- longitud: double (nullable = true)
 |-- fecha_instalacion: date (nullable = true)


Primeros registros:
+-----------+------------------------------------------+----------+-------------+---------------------+---------+---------+-----------------+
|estacion_id|nombre_estacion                           |region    |cuenca       |tipo_fuente          |latitud  |longitud |fecha_instalacion|
+-----------+------------------------------------------+----------+-------------+---------------------+---------+---------+-----------------+
|EST-0001   |Estación Río Río Coata 1                  

In [54]:
col("ubicacion") == "Rio Coata"


Column<'=(ubicacion, 'Rio Coata')'>

In [57]:
# ============================================================
# 11. ESCRITURA PARTICIONADA EN PARQUET → GOLD
# ============================================================

from pyspark.sql.functions import col, trim

# ------------------------------------------------------------
# 1. Preparar el catálogo de estaciones
# ------------------------------------------------------------

df_estaciones_limpio = df_estaciones.select(
    trim(col("estacion_id")).alias("estacion_id"),
    trim(col("nombre_estacion")).alias("ubicacion")
).dropDuplicates(["estacion_id"])

# ------------------------------------------------------------
# 2. Unir las mediciones con la información de estaciones
# ------------------------------------------------------------

df_agua_gold = df_agua_valido.join(
    df_estaciones_limpio,
    on="estacion_id",
    how="left"
)

# ------------------------------------------------------------
# 3. Verificar que se creó la ubicación
# ------------------------------------------------------------

print("Columnas de df_agua_gold:")
print(df_agua_gold.columns)

print("\nEjemplos de estaciones y ubicaciones:")
df_agua_gold.select(
    "estacion_id",
    "ubicacion"
).distinct().show(20, truncate=False)

# ------------------------------------------------------------
# 4. Verificar ubicaciones nulas
# ------------------------------------------------------------

nulos_ubicacion = df_agua_gold.filter(
    col("ubicacion").isNull()
).count()

print(f"\nRegistros sin ubicación: {nulos_ubicacion}")

# ------------------------------------------------------------
# 5. Escritura particionada en Parquet
# ------------------------------------------------------------

ruta_gold = f"{ARTIFACTS}/lecturas_particionado"

(
    df_agua_gold
    .repartition(4)
    .write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("ubicacion")
    .save(ruta_gold)
)

print("\n============================================")
print("GOLD creada correctamente")
print(f"Ruta: {ruta_gold}")
print("============================================")

# ------------------------------------------------------------
# 6. Verificar carpetas de partición
# ------------------------------------------------------------

import os

print("\nCarpetas de partición:")

for carpeta in sorted(os.listdir(ruta_gold)):
    if carpeta.startswith("ubicacion="):
        print(carpeta)


Columnas de df_agua_gold:
['estacion_id', 'medicion_id', 'fecha', 'hora', 'canal', 'ph', 'temperatura_c', 'turbidez_ntu', 'oxigeno_disuelto_mgl', 'conductividad_us_cm', 'solidos_disueltos_ppm', 'nitratos_mgl', 'fosfatos_mgl', 'coliformes_fecales_cfu', 'observaciones', 'ubicacion']

Ejemplos de estaciones y ubicaciones:
+-----------+------------------------------------------------+
|estacion_id|ubicacion                                       |
+-----------+------------------------------------------------+
|EST-0133   |Estación Río Río Amazonas 133                   |
|EST-0153   |Estación Planta de tratamiento Río Vilcanota 153|
|EST-0140   |Estación Río Río Coata 140                      |
|EST-0143   |Estación Lago Río Chancay 143                   |
|EST-0179   |Estación Río Río Chancay 179                    |
|EST-0167   |Estación Lago Río Piura 167                     |
|EST-0159   |Estación Pozo subterráneo Río Apurímac 159      |
|EST-0112   |Estación Pozo subterráneo Río Nanay 


GOLD creada correctamente
Ruta: /opt/s03-procesamiento-calidad-datos/artifacts/lecturas_particionado

Carpetas de partición:
ubicacion=Estación Lago Lago Junín 43
ubicacion=Estación Lago Lago Junín 7
ubicacion=Estación Lago Lago Titicaca 128
ubicacion=Estación Lago Río Apurímac 94
ubicacion=Estación Lago Río Chancay 101
ubicacion=Estación Lago Río Chancay 117
ubicacion=Estación Lago Río Chancay 143
ubicacion=Estación Lago Río Chancay 173
ubicacion=Estación Lago Río Chancay 64
ubicacion=Estación Lago Río Chancay 92
ubicacion=Estación Lago Río Chancay 99
ubicacion=Estación Lago Río Chili 141
ubicacion=Estación Lago Río Chili 155
ubicacion=Estación Lago Río Chillón 151
ubicacion=Estación Lago Río Chillón 37
ubicacion=Estación Lago Río Chillón 44
ubicacion=Estación Lago Río Chira 12
ubicacion=Estación Lago Río Chira 176
ubicacion=Estación Lago Río Coata 174
ubicacion=Estación Lago Río Ilave 175
ubicacion=Estación Lago Río Ilave 77
ubicacion=Estación Lago Río Ilave 80
ubicacion=Estación La

In [58]:
col("ubicacion") == "Rio Coata"


Column<'=(ubicacion, 'Rio Coata')'>

In [59]:
# ============================================================
# 12. LEER DE VUELTA Y VERIFICAR
# ============================================================

# Ruta del Parquet particionado
ruta_gold = f"{ARTIFACTS}/lecturas_particionado"

# ------------------------------------------------------------
# 1. Leer nuevamente el Parquet particionado
# ------------------------------------------------------------

df_verificacion = spark.read.parquet(ruta_gold)

print("ESQUEMA DEL PARQUET PARTICIONADO:")
df_verificacion.printSchema()

# ------------------------------------------------------------
# 2. Comparar cantidad de registros
# ------------------------------------------------------------

total_verificacion = df_verificacion.count()
total_original = df_agua_valido.count()

print(f"\nRegistros originales:  {total_original}")
print(f"Registros recuperados: {total_verificacion}")

assert total_verificacion == total_original

print("\nVerificacion OK: el conteo coincide.")

# ------------------------------------------------------------
# 3. Revisar plan de ejecución
# ------------------------------------------------------------
# Usamos una ubicación REAL existente en nuestro dataset

ubicacion_prueba = df_verificacion.select(
    "ubicacion"
).where(
    col("ubicacion").isNotNull()
).first()[0]

print("\n============================================")
print("Ubicacion utilizada para probar el filtro:")
print(ubicacion_prueba)
print("============================================")

print("\nPLAN DE EJECUCION:")
df_verificacion.filter(
    col("ubicacion") == ubicacion_prueba
).explain(True)

# ------------------------------------------------------------
# 4. Balance de registros por partición
# ------------------------------------------------------------

print("\n============================================")
print("BALANCE DE REGISTROS POR UBICACION")
print("============================================")

df_verificacion \
    .groupBy("ubicacion") \
    .count() \
    .orderBy("ubicacion") \
    .show(truncate=False)

# ------------------------------------------------------------
# 5. Liberar memoria
# ------------------------------------------------------------

df_agua_valido.unpersist()

print("============================================")
print("Proceso de verificacion finalizado.")
print("============================================")


ESQUEMA DEL PARQUET PARTICIONADO:
root
 |-- estacion_id: string (nullable = true)
 |-- medicion_id: long (nullable = true)
 |-- fecha: string (nullable = true)
 |-- hora: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- ph: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- oxigeno_disuelto_mgl: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_ppm: double (nullable = true)
 |-- nitratos_mgl: double (nullable = true)
 |-- fosfatos_mgl: double (nullable = true)
 |-- coliformes_fecales_cfu: double (nullable = true)
 |-- observaciones: string (nullable = true)
 |-- ubicacion: string (nullable = true)




Registros originales:  1250000
Registros recuperados: 1250000

Verificacion OK: el conteo coincide.

Ubicacion utilizada para probar el filtro:
Estación Lago Lago Titicaca 128

PLAN DE EJECUCION:
== Parsed Logical Plan ==
'Filter '`=`('ubicacion, Estación Lago Lago Titicaca 128)
+- Relation [estacion_id#5884,medicion_id#5885L,fecha#5886,hora#5887,canal#5888,ph#5889,temperatura_c#5890,turbidez_ntu#5891,oxigeno_disuelto_mgl#5892,conductividad_us_cm#5893,solidos_disueltos_ppm#5894,nitratos_mgl#5895,fosfatos_mgl#5896,coliformes_fecales_cfu#5897,observaciones#5898,ubicacion#5899] parquet

== Analyzed Logical Plan ==
estacion_id: string, medicion_id: bigint, fecha: string, hora: string, canal: string, ph: double, temperatura_c: double, turbidez_ntu: double, oxigeno_disuelto_mgl: double, conductividad_us_cm: double, solidos_disueltos_ppm: double, nitratos_mgl: double, fosfatos_mgl: double, coliformes_fecales_cfu: double, observaciones: string, ubicacion: string
Filter (ubicacion#5899 = Estac

[Stage 200:>                                                      (0 + 24) / 24]

+-------------------------------+-----+
|ubicacion                      |count|
+-------------------------------+-----+
|Estación Lago Lago Junín 43    |6871 |
|Estación Lago Lago Junín 7     |6761 |
|Estación Lago Lago Titicaca 128|7024 |
|Estación Lago Río Apurímac 94  |7085 |
|Estación Lago Río Chancay 101  |6934 |
|Estación Lago Río Chancay 117  |7109 |
|Estación Lago Río Chancay 143  |6936 |
|Estación Lago Río Chancay 173  |6960 |
|Estación Lago Río Chancay 64   |6906 |
|Estación Lago Río Chancay 92   |7113 |
|Estación Lago Río Chancay 99   |6954 |
|Estación Lago Río Chili 141    |7053 |
|Estación Lago Río Chili 155    |7014 |
|Estación Lago Río Chillón 151  |7044 |
|Estación Lago Río Chillón 37   |6891 |
|Estación Lago Río Chillón 44   |7081 |
|Estación Lago Río Chira 12     |6898 |
|Estación Lago Río Chira 176    |6825 |
|Estación Lago Río Coata 174    |6964 |
|Estación Lago Río Ilave 175    |7096 |
+-------------------------------+-----+
only showing top 20 rows
Proceso de veri

Capas de mi pipeline (Proyecto Sello - Agua)

Bronze (raw): archivo Parquet de mediciones de calidad del agua, cargado desde la fuente original sin aplicar transformaciones de limpieza.

Silver: df_agua_valido — datos procesados mediante validaciones, tratamiento de valores nulos y eliminación de registros duplicados cuando corresponde. Se conservan las columnas reales del dataset, como medicion_id, estacion_id, fecha, hora, ph, temperatura_c, turbidez_ntu, oxigeno_disuelto_mgl, conductividad_us_cm, solidos_disueltos_ppm, nitratos_mgl, fosfatos_mgl y coliformes_fecales_cfu.

Gold: conjunto Parquet particionado por ubicacion, preparado para su posterior consumo en análisis, dashboards o procesos de Machine Learning distribuido.

In [60]:
# Técnica 1 — distinct() / dropDuplicates()

total = df_agua.count()

# Duplicados considerando toda la fila
sin_dup_fila_completa = df_agua.distinct().count()

# Duplicados según el identificador de medición
sin_dup_por_medicion = df_agua.dropDuplicates(["medicion_id"]).count()

# Duplicados según estación + fecha + hora
sin_dup_estacion_fecha_hora = df_agua.dropDuplicates(
    ["estacion_id", "fecha", "hora"]
).count()

print(f"Total: {total}")
print(f"Sin duplicar (fila completa): {sin_dup_fila_completa}")
print(f"Sin duplicar (por medicion_id): {sin_dup_por_medicion}")
print(f"Sin duplicar (por estacion_id+fecha+hora): {sin_dup_estacion_fecha_hora}")


[Stage 218:===============================================>         (5 + 1) / 6]

Total: 1250000
Sin duplicar (fila completa): 1250000
Sin duplicar (por medicion_id): 1250000
Sin duplicar (por estacion_id+fecha+hora): 1245817


In [61]:
print(
    "Filas si se usa na.drop() sin argumentos:",
    df_agua.na.drop().count()
)


Filas si se usa na.drop() sin argumentos: 150152


In [62]:
df_agua_valido = df_agua.na.drop(
    subset=["medicion_id", "estacion_id"]
)

print(f"Filas antes: {df_agua.count()}")
print(
    f"Filas después de na.drop(subset=['medicion_id', 'estacion_id']): "
    f"{df_agua_valido.count()}"
)


Filas antes: 1250000
Filas después de na.drop(subset=['medicion_id', 'estacion_id']): 1250000


In [63]:
assert df_agua_valido.filter(
    col("medicion_id").isNull()
).count() == 0

assert df_agua_valido.filter(
    col("estacion_id").isNull()
).count() == 0

print("Validación OK: no existen mediciones sin identificador ni estación.")


Validación OK: no existen mediciones sin identificador ni estación.


In [64]:
df_agua_valido = df_agua_valido.cache()

print("Silver creada y almacenada en caché.")


Silver creada y almacenada en caché.


Capas de mi pipeline (Proyecto Sello - Agua)

Bronze (raw): archivo de mediciones de calidad del agua cargado desde la fuente original, sin aplicar transformaciones de limpieza.

Silver: df_agua_valido — datos procesados mediante validaciones de calidad y tratamiento de nulos. Se conservan las columnas reales del dataset, como medicion_id, estacion_id, fecha, hora, canal, ph, temperatura_c, turbidez_ntu, oxigeno_disuelto_mgl, conductividad_us_cm, solidos_disueltos_ppm, nitratos_mgl, fosfatos_mgl y coliformes_fecales_cfu.

Gold: salida Parquet preparada para el análisis posterior y el consumo por procesos de análisis, dashboards o Machine Learning distribuido.